In [2]:
from dotenv import load_dotenv
import os
from pathlib import Path
load_dotenv()
PATH_FINTOC_2022 = Path(os.getenv('PATH_FINTOC_2022'))

In [2]:
FINTOC_EN = PATH_FINTOC_2022/'data'/'en'
FINTOC_EN_PDF =FINTOC_EN/'pdf'
FINTOC_EN_ANNOTS = FINTOC_EN/'annots'
FINTOC_EN_PRED = FINTOC_EN/'preds'

JSON_EXTENSION = "fintoc4.json"

In [3]:
if not FINTOC_EN_PRED.exists():
    FINTOC_EN_PRED.mkdir()
FILE_PATHS = list(FINTOC_EN_PDF.iterdir())[:10]

In [4]:
import json

import pager
from pager.pager_pipe_line import pdf2region_row

from pager.doc_model import MinerPDFModel
from pager.doc_model import LogicTreeModel
from pager.doc_model.converters import PDF2LogicTree
from pager.doc_model.dtype import Document, Page
from pager import Region

from typing import Dict

pdf_reader = MinerPDFModel({"page_model": pdf2region_row})
pdf2tree = PDF2LogicTree()
tree = LogicTreeModel()

In [5]:
file_path = FILE_PATHS[0]
pdf_reader.read_from_file(file_path)
pdf_reader.extract()
pdf2tree.convert(pdf_reader, tree)
file_path

/home/daniil/project/PageR/env/lib/python3.12/site-packages/torch_geometric/nn/conv/tag_conv.py:102: UserWarning: Converting sparse tensor to CSR format for more efficient processing. Consider converting your sparse tensor to CSR format beforehand to avoid repeated conversion (got 'torch.sparse_coo')
  return spmm(adj_t, x, reduce=self.aggr)
/home/daniil/project/PageR/env/lib/python3.12/site-packages/torch_geometric/utils/_spmm.py:71: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  src = src.to_sparse_csr()
/home/daniil/project/PageR/env/lib/python3.12/site-packages/torch/nn/modules/module.py:1775: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


PosixPath('/home/dataset/fintoc2022/data/en/pdf/LU0066480616-LU0208183011_English_2012_ValartisRussianMarketFund.pdf')

# Отладка на одном документе

In [6]:
doc_rows = [row for page in tree.document.pages for row in page.rows]
row_num_page = [page.num_page for page in tree.document.pages for _ in page.rows]

In [7]:
# for row in doc_rows:
#     print(row.text)

### Кластер строк

In [8]:
_font_from_doc = {}
_list_font_type = []
def get_vec_feature(row):
    fd = row.font.to_dict()
    name = fd['name']
    size = int(round(fd['size']/2))
    width = int(round(fd['width']/0.1))
    italic = int(round(fd['italic']/0.1))
    return (name, size, width, italic)

def get_id_exist_cluster(feature):
    _list_font_type.append(feature)
    if feature in _font_from_doc.keys():
         return _font_from_doc[feature] 
    return None
        
def add_new_cluster(feature):
    indexs= list(_font_from_doc.values())
    index = max(indexs)+1 if len(indexs) != 0 else 0
    _font_from_doc[feature] = index
    clusters[index] = []
    return index
    
clusters = {}
for row in doc_rows:
    feature_row = get_vec_feature(row)
    index = get_id_exist_cluster(feature_row)
    if index is None:
        index = add_new_cluster(feature_row)
    clusters[index].append(row)

### Поиск кластера основного текста

In [9]:
max_clust = -1
max_size = 0
for ind, rows in clusters.items():
    if max_size < len(rows):
        max_clust = ind
        max_size = len(rows)
font_text =  clusters[max_clust][0].font
print(max_size)
font_text.to_dict()

1569


{'name': 'Helvetica', 'width': 0.0, 'italic': 0.0, 'size': 9.694196122319966}

### Поиск кластеров заголовков

In [10]:
header_cluster = {ind: cl[0] for ind, cl in clusters.items() if cl[0].font > font_text}

### Сортировка кластеров заголовков

In [11]:
header_cluster_index = list(header_cluster.keys())
header_cluster_index.sort(key=lambda ind: header_cluster[ind].font, reverse=True)
for header_ind in header_cluster_index:
    print(get_vec_feature(header_cluster[header_ind]))

level_header_cluster = {cl:level+1 for level, cl in enumerate(header_cluster_index)}
# level_header_cluster

('Helvetica-Bold', 9, 10, 0)
('Helvetica-Bold', 8, 10, 0)
('Helvetica-Bold', 7, 10, 0)
('Helvetica-Bold', 6, 10, 0)
('Helvetica-Bold', 5, 10, 0)
('Helvetica-BoldOblique', 5, 10, 0)
('Helvetica-Bold', 4, 10, 0)


### Создание регионов

In [12]:
num_cluster = [ _font_from_doc[vec] for vec in _list_font_type]
regions = []

def get_region(row, cl, num_page):
    tmp_reg = {
        "rows": [row],
        "label": "header" if cl in level_header_cluster else "text",
        "metainfo": {"page": num_page}
    }
    if cl in level_header_cluster:
        tmp_reg["header_level"] = level_header_cluster[cl]
    return tmp_reg

tmp_cl = num_cluster[0]
tmp_num_page = 0
tmp_reg = get_region(rows[0].to_dict(), tmp_cl, tmp_num_page)
for row, cl, num_page in zip(doc_rows[1:], num_cluster[1:], row_num_page[1:]):
    if cl == tmp_cl and num_page==tmp_num_page:
        tmp_reg['rows'].append(row.to_dict())
    else:
        regions.append(tmp_reg.copy())
        tmp_cl = cl
        tmp_num_page = num_page
        tmp_reg = get_region(row.to_dict(), cl,num_page)

### Построение дерева (Переиспользование класса конвертера)

In [13]:


class CustomJson2LogicTree(PDF2LogicTree):
    def upload(self, regions, output_model:LogicTreeModel):
        reg_page = [[]]
        tmp_page = 0
        for reg in regions:
            if tmp_page == reg['metainfo']["page"]:
                reg_page[tmp_page].append(reg)
            else:
                reg_page.append([reg])
                tmp_page += 1
            
        document = Document([Page(regions=[Region(r) for r in regs], num_page=num) for num, regs in enumerate(reg_page)])
        regions = [reg for page in document.pages for reg in page.regions]
        tree = self.get_tree_from_doc(regions)
        self.set_level(regions, tree['parent'])
        
        output_model.document = document
        output_model.regions = regions
        output_model.edges = tree


creater = CustomJson2LogicTree()
tree = LogicTreeModel()
creater.upload(regions, tree)

In [14]:
print(tree.document.md)

# Distributor).    The  Distributor  is  entitled  to  delegate  its  functions  to  one  or  more  other  persons 

L'apposition du visa ne peut en aucun cas servir d'argument de publicité Luxembourg, le 2012-10-08 Commission de Surveillance du Secteur Financier

# VALARTIS RUSSIAN MARKET FUND 

an investment company with variable capital (société d’investissement à capital variable) organised as an undertakings for collective investment in transferable securities (organisme de placement collectif en valeurs mobilières) incorporated in and under the laws of the Grand Duchy of Luxembourg

VAT Registration No. LU21668963

## PROSPECTUS 

### 25 SEPTEMBER 2012 


<!-- page_num: 0 --> 
---### CONTENTS 

1.  MANAGEMENT AND ADMINISTRATION ............................................................................................. 6 2.  DEFINITIONS ........................................................................................................................................ 8 3.  P

# Упаковка в один класс

In [6]:
class CustomJson2LogicTree(PDF2LogicTree):
    def upload(self, regions, output_model:LogicTreeModel):
        reg_page = [[]]
        tmp_page = 0
        for reg in regions:
            if tmp_page == reg['metainfo']["page"]:
                reg_page[tmp_page].append(reg)
            else:
                reg_page.append([reg])
                tmp_page += 1
            
        document = Document([Page(regions=[Region(r) for r in regs], num_page=num) for num, regs in enumerate(reg_page)])
        regions = [reg for page in document.pages for reg in page.regions]
        tree = self.get_tree_from_doc(regions)
        self.set_level(regions, tree['parent'])
        
        output_model.document = document
        output_model.regions = regions
        output_model.edges = tree

class ExtractTree:
    def __init__(self):
        self.creater = CustomJson2LogicTree()
        self._font_from_doc = {}
        self._list_font_type = []

        
    def extract(self, tree:LogicTreeModel):
        doc_rows = [row for page in tree.document.pages for row in page.rows]
        row_num_page = [page.num_page for page in tree.document.pages for _ in page.rows]
        clusters = self.get_cluster(doc_rows)
        level_header_cluster = self.get_level_header(clusters)
        regions = self.get_regions(doc_rows, row_num_page, level_header_cluster)
        self.creater.upload(regions, tree)


    def get_regions(self, doc_rows, row_num_page, level_header_cluster):
        num_cluster = [ self._font_from_doc[vec] for vec in self._list_font_type]
        regions = []
        
        def get_region(row, cl, num_page):
            tmp_reg = {
                "rows": [row],
                "label": "header" if cl in level_header_cluster else "text",
                "metainfo": {"page": num_page}
            }
            if cl in level_header_cluster:
                tmp_reg["header_level"] = level_header_cluster[cl]
            return tmp_reg
        
        tmp_cl = num_cluster[0]
        tmp_num_page = 0
        tmp_reg = get_region(doc_rows[0].to_dict(), tmp_cl, tmp_num_page)
        for row, cl, num_page in zip(doc_rows[1:], num_cluster[1:], row_num_page[1:]):
            if cl == tmp_cl and num_page==tmp_num_page:
                tmp_reg['rows'].append(row.to_dict())
            else:
                regions.append(tmp_reg.copy())
                tmp_cl = cl
                tmp_num_page = num_page
                tmp_reg = get_region(row.to_dict(), cl,num_page)
        return regions
        
    def get_level_header(self, clusters):
        main_clust, main_font = self.get_main_cluster(clusters)
        header_cluster = {ind: cl[0] for ind, cl in clusters.items() if cl[0].font > main_font}
        header_cluster_index = list(header_cluster.keys())
        header_cluster_index.sort(key=lambda ind: header_cluster[ind].font, reverse=True)        
        level_header_cluster = {cl:level+1 for level, cl in enumerate(header_cluster_index)}
        return level_header_cluster
        
    def get_main_cluster(self, clusters):
        max_clust = -1
        max_size = 0
        for ind, rows in clusters.items():
            if max_size < len(rows):
                max_clust = ind
                max_size = len(rows)
        font_text =  clusters[max_clust][0].font

        font_text.to_dict()
        return max_clust, font_text

    def get_cluster(self, doc_rows):
        self._font_from_doc = {}
        self._list_font_type = []
        def get_vec_feature(row):
            fd = row.font.to_dict()
            name = fd['name']
            size = int(round(fd['size']/2))
            width = int(round(fd['width']/0.1))
            italic = int(round(fd['italic']/0.1))
            return (name, size, width, italic)
        
        def get_id_exist_cluster(feature):
            self._list_font_type.append(feature)
            if feature in self._font_from_doc.keys():
                 return self._font_from_doc[feature] 
            return None
                
        def add_new_cluster(feature):
            indexs= list(self._font_from_doc.values())
            index = max(indexs)+1 if len(indexs) != 0 else 0
            self._font_from_doc[feature] = index
            clusters[index] = []
            return index
            
        clusters = {}
        for row in doc_rows:
            feature_row = get_vec_feature(row)
            index = get_id_exist_cluster(feature_row)
            if index is None:
                index = add_new_cluster(feature_row)
            clusters[index].append(row)
        return clusters
        
       
     
   

extract_tree=ExtractTree() 

In [7]:
def tree2fintoc_json(doc_tree) -> Dict:
    fintoc_json = []
    for id_node, node in doc_tree['nodes']['regions'].items():
        if node['label'] == 'header':
            reg = {
                "text":node['text'],
                "id": id_node,
                "page": node['metainfo']['page'] + 1, # В PageR специально оставил нумерацию с нуля (поскольку номер определяется не порядком, а пишится на странице и это еще не реализовано)
                "depth": node['header_level']
            }
            fintoc_json.append(reg)
    return fintoc_json

N = len(FILE_PATHS)
for i, file_path in enumerate(FILE_PATHS):    
    pdf_reader.read_from_file(file_path)
    pdf_reader.extract()
    pdf2tree.convert(pdf_reader, tree)
    extract_tree.extract(tree)
    
    doc_tree = tree.to_dict()
    pred_path = FINTOC_EN_PRED/(file_path.name+'.'+JSON_EXTENSION)
    # print(tree.document.md)
    
    rez = tree2fintoc_json(doc_tree)
    with open(pred_path, 'w') as f:
        json.dump(rez, f)

    print(f'{i+1}/{N} ({(i+1)/N*100:.2f}%)')

1/10 (10.00%)
2/10 (20.00%)
3/10 (30.00%)
4/10 (40.00%)


Cannot set gray non-stroke color because /'P0' is an invalid float value


5/10 (50.00%)
6/10 (60.00%)
7/10 (70.00%)
8/10 (80.00%)
9/10 (90.00%)
10/10 (100.00%)


In [8]:
import subprocess
import pandas as pd

In [9]:
subprocess.run(['python', 'metric.py', '--gt_folder', FINTOC_EN_ANNOTS, '--submission_folder', FINTOC_EN_PRED])
df = pd.read_csv('td_report.csv', delimiter='\t')
df[-2:-1][['Prec', 'Rec', 'F1']]

,Prec,Rec,F1
49,0.46105189050171447,0.6057079550435704,0.4664353285809487


In [10]:
df = pd.read_csv('toc_report.csv', delimiter='\t')
tbl = df[-4:-3]
tbl[['Inex08-P', 'Inex08-R', 'Inex08-F1', 'Inex08-Title acc', 'Inex08-Level acc']]

,Inex08-P,Inex08-R,Inex08-F1,Inex08-Title acc,Inex08-Level acc
55,30.2,31.4,28.0,45.3,33.7
